In [5]:
import pandas as pd
from transformers import T5Tokenizer,Trainer, TrainingArguments, T5ForConditionalGeneration
import re



In [52]:
import pandas as pd

train_data = pd.read_csv("/content/samsum-train.csv")
val_data = pd.read_csv("/content/samsum-validation.csv")

In [53]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [54]:
train_data.shape

(14732, 3)

In [55]:
val_data.shape

(818, 3)

In [56]:
# randompy sampling
train_data = train_data.sample(n=5000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)


In [57]:
train_data.shape

(5000, 3)

In [58]:
# Data Pre-processing

import re

def clean_data(text):
  text = re.sub(r"\r\n", " ", text) # lines removed
  text = re.sub(r"\s+", " ", text) # spaces removed (corrected regex)
  text = re.sub(r"<.*?>", " ", text) # html tags are removed
  text = text.strip().lower()
  return text

In [59]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

In [60]:
# Tokanizer

tokanizer = T5Tokenizer.from_pretrained('t5-small')


In [61]:
# row data into tokanizers inputs for fine tuning

def tokanize(data):
  max_length = 512 # Define max_length
  inputs = tokanizer(data['dialogue'], padding='max_length', max_length=max_length, truncation=True)
  target = tokanizer(data['summary'], padding='max_length', max_length=150, truncation=True)

  inputs['labels'] = target["input_ids"]  # tokens id's => add to imput as labels
  return inputs

In [62]:
train_dataset = train_data.apply(tokanize, axis=1).tolist()
val_dataset = val_data.apply(tokanize, axis=1).tolist()

In [63]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [64]:
len(train_dataset[0]['input_ids'])

512

In [65]:
# start working with the model

model = T5ForConditionalGeneration.from_pretrained('t5-small')

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [66]:
# Fine Tuning the model

import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ", device)


device:  cuda


In [67]:
# Training arguements

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    warmup_steps=500

)

In [68]:
trainer = Trainer(
    model= model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [69]:
# Training Model

model = trainer.train()

Epoch,Training Loss,Validation Loss
1,3.616980,0.371134
2,0.399100,0.354948
3,0.378127,0.349139
4,0.360921,0.346488
5,0.355162,0.345271
6,0.349735,0.345112


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [70]:
# model load => fine-tuning => save the model



# save the model
trainer.model.save_pretrained('./saved_summary_model')
tokanizer.save_pretrained('./saved_summary_model')

print("Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


In [71]:

from transformers import T5ForConditionalGeneration, T5Tokenizer


# Load the model
model = T5ForConditionalGeneration.from_pretrained('./saved_summary_model')
tokenizer = T5Tokenizer.from_pretrained('t5-small')  # ← use t5-small, not saved model


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [72]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)  # clean

    # Tokenize
    inputs = tokenizer(
        dialogue,
        max_length=512,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)  # move inputs to device here

    # Generate summary token ids
    model.to(device)
    target = model.generate(
      input_ids=inputs['input_ids'],
      attention_mask=inputs['attention_mask'],
      max_length=200,       # increase
      min_length=60,        # force longer summary
      num_beams=4,
      length_penalty=2.0,   # increase to encourage longer output
      early_stopping=True
  )

    # Decode token ids to text
    summary = tokenizer.decode(target[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)

    return summary

In [73]:
test_dialouge = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance.
Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experience.
Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks.
Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on.
Reporter: Governments and organizations are beginning to introduce regulations to guide the development of AI.
Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, are difficult to interpret.
Reporter: Experts also highlight the importance of responsible AI development, including data privacy.
Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be essential.
"""
print(summarize_dialogue(test_dialouge))


ai systems are becoming more capable due to advances in deep learning and access to large datasets. experts highlight the importance of responsible ai development, including data privacy. ai systems are difficult to interpret, but they often reflect the data they are trained on. experts also highlight the importance of responsible ai development.


In [ ]:
print('hello')